In [ ]:
import numpy as np
import time
from scipy.optimize import linprog

In [ ]:
# Матрица затрат C (3×3): строки — поставщики P1,P2,P3; столбцы — устройства D1,D2,D3
C = np.array([
    [120, 130, 140],  # P1
    [135, 125, 138],  # P2
    [150, 145, 132],  # P3
], dtype=np.float64)

print("Матрица затрат C (руб./ед.):")
print(f"{'':>5} {'D1':>6} {'D2':>6} {'D3':>6}")
for i, row in enumerate(C, 1):
    print(f"P{i}:  {row[0]:>6.0f} {row[1]:>6.0f} {row[2]:>6.0f}")

In [ ]:
# Вектор предложения: лимиты поставщиков P1, P2, P3
b_ub = np.array([5000, 4000, 3500], dtype=np.float64)
print("b_ub (лимиты поставщиков):", b_ub)

In [ ]:
# Вектор спроса на устройства D1, D2, D3
b_eq = np.array([3000, 4200, 2500], dtype=np.float64)
print("b_eq (спрос на устройства):", b_eq)
print(f"Суммарный спрос: {b_eq.sum():.0f}  |  Суммарное предложение: {b_ub.sum():.0f}  → задача допустима")

In [ ]:
# Разворачиваем матрицу C построчно: [c11,c12,c13, c21,c22,c23, c31,c32,c33]
c = C.flatten()
labels = [f"x{i}{j}" for i in range(1,4) for j in range(1,4)]
print("Вектор c:", dict(zip(labels, c.astype(int))))

In [ ]:
# Каждая строка — один поставщик
# x11 x12 x13 x21 x22 x23 x31 x32 x33
A_ub = np.array([
    [1,  1,  1,  0,  0,  0,  0,  0,  0],  # P1: x11+x12+x13 <= 5000
    [0,  0,  0,  1,  1,  1,  0,  0,  0],  # P2: x21+x22+x23 <= 4000
    [0,  0,  0,  0,  0,  0,  1,  1,  1],  # P3: x31+x32+x33 <= 3500
], dtype=np.float64)

print("Матрица A_ub (3×9):")
print(f"{'':>4}", '  '.join(f"{l:>4}" for l in labels), "  b_ub")
for i, (row, b) in enumerate(zip(A_ub, b_ub), 1):
    print(f"P{i}: ", '  '.join(f"{int(v):>4}" for v in row), f"  {b:.0f}")

In [ ]:
# Каждая строка — одно устройство
# x11 x12 x13 x21 x22 x23 x31 x32 x33
A_eq = np.array([
    [1,  0,  0,  1,  0,  0,  1,  0,  0],  # D1: x11+x21+x31 = 3000
    [0,  1,  0,  0,  1,  0,  0,  1,  0],  # D2: x12+x22+x32 = 4200
    [0,  0,  1,  0,  0,  1,  0,  0,  1],  # D3: x13+x23+x33 = 2500
], dtype=np.float64)

print("Матрица A_eq (3×9):")
print(f"{'':>4}", '  '.join(f"{l:>4}" for l in labels), "  b_eq")
for j, (row, b) in enumerate(zip(A_eq, b_eq), 1):
    print(f"D{j}: ", '  '.join(f"{int(v):>4}" for v in row), f"  {b:.0f}")

In [ ]:
methods = ['highs', 'highs-ds', 'highs-ipm']
results = {}
times = {}

for method in methods:
    t0 = time.perf_counter()
    res = linprog(
        c=c,
        A_ub=A_ub, b_ub=b_ub,
        A_eq=A_eq, b_eq=b_eq,
        bounds=(0, None),
        method=method
    )
    dt = time.perf_counter() - t0
    results[method] = res
    times[method] = dt
    print(f"Метод '{method}': статус={'Успех' if res.success else 'Ошибка'}  "
          f"Z = {res.fun:.2f} руб.  время = {dt*1000:.4f} мс")

In [ ]:
print(f"{'Метод':<12} {'Статус':<10} {'Z, руб.':<18} {'Время, мс'}")
print("-" * 50)
for m in methods:
    r = results[m]
    print(f"{m:<12} {'Успех' if r.success else 'Ошибка':<10} {r.fun:<18.2f} {times[m]*1000:.4f}")

fastest = min(times, key=times.get)
print(f"\nСамый быстрый метод: '{fastest}' ({times[fastest]*1000:.4f} мс)")

In [ ]:
res = results['highs']
x = res.x.reshape(3, 3)
suppliers = ['P1', 'P2', 'P3']
devices   = ['D1', 'D2', 'D3']

print("Оптимальный план закупок X* (ед./мес.):")
print(f"{'':>5} {'D1':>8} {'D2':>8} {'D3':>8} {'Итого':>8}")
for i, sup in enumerate(suppliers):
    print(f"{sup}:  {x[i,0]:>8.1f} {x[i,1]:>8.1f} {x[i,2]:>8.1f} {x[i].sum():>8.1f}  (лимит: {b_ub[i]:.0f})")
print(f"{'Спрос':>5}: {'3000':>8} {'4200':>8} {'2500':>8}")

print("\nНенулевые поставки:")
for i, sup in enumerate(suppliers):
    for j, dev in enumerate(devices):
        if x[i, j] > 0.01:
            print(f"  {sup} → {dev}: {x[i,j]:.0f} ед. × {C[i,j]:.0f} руб. = {x[i,j]*C[i,j]:.0f} руб.")

print(f"\nМинимальные суммарные затраты: {res.fun:.2f} руб./мес.")